# Week 2 — EDA: Reading the City in Pixels (STARTER)
### The Computer Vision Internship | VisionAI Lagos

Amara's brief: by Friday you need a two-page EDA summary — are the four cities spectrally separable, which classes confuse each other, and do train/val/test splits have consistent statistics?

**Before any code — write your hypothesis:**

## My Hypothesis

**Will city clusters separate more or less cleanly than class clusters in PCA?** [WRITE HERE]

**Which class boundary will cause the most confusion?** [WRITE HERE]

**What does the answer imply for cross-city deployment?** [WRITE HERE]

## Part 1 — Distribution and Class Balance

In [ ]:
print(f"Class distribution:"); print(df["land_use"].value_counts())
print(f"\nCity distribution:"); print(df["city"].value_counts())
print(f"\nSplit distribution:"); print(df["split"].value_counts())

fig,axes=plt.subplots(1,2,figsize=(13,5))
lu_counts=df["land_use"].value_counts()
axes[0].bar(lu_counts.index,lu_counts.values,color=[CLASS_CLR[k] for k in lu_counts.index])
for bar,c in zip(axes[0].patches,lu_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2,bar.get_height()+20,f"{c:,}",ha="center",fontsize=9)
axes[0].set_title("Land-Use Class Distribution",fontweight="bold",loc="left")
axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)
pivot=df.groupby(["city","land_use"]).size().unstack()
sns.heatmap(pivot,annot=True,fmt="d",cmap="Blues",ax=axes[1],linewidths=0.5,linecolor="white")
axes[1].set_title("City × Class Distribution",fontweight="bold",loc="left")
plt.tight_layout(); plt.savefig("outputs/class_distribution.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

## Part 2 — Spectral EDA

> ⏸️ **Pause and Predict**
> Before running: which city do you expect to separate most clearly in the R vs G scatter plot — Kano (arid) or Ibadan (humid)? Write your prediction.

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,10))
# 1. Brightness violin by city
city_order=df.groupby("city")["brightness"].mean().sort_values(ascending=False).index
parts=axes[0][0].violinplot([df[df["city"]==c]["brightness"].values for c in city_order],
                              positions=range(len(city_order)),showmedians=True)
for pc,city in zip(parts["bodies"],city_order):
    pc.set_facecolor(CITY_CLR[city]); pc.set_alpha(0.7)
axes[0][0].set_xticks(range(len(city_order))); axes[0][0].set_xticklabels(city_order)
axes[0][0].set_title("Brightness by City",fontweight="bold",loc="left")
axes[0][0].set_ylabel("Mean pixel brightness")
axes[0][0].spines["top"].set_visible(False); axes[0][0].spines["right"].set_visible(False)
# 2. NDVI by class
for lu,grp in df.groupby("land_use"):
    axes[0][1].hist(grp["ndvi_proxy"],bins=40,alpha=0.5,label=lu,color=CLASS_CLR[lu],density=True)
axes[0][1].axvline(0,color="black",linewidth=1,linestyle="--")
axes[0][1].set_title("NDVI Proxy by Class",fontweight="bold",loc="left")
axes[0][1].legend(fontsize=9); axes[0][1].spines["top"].set_visible(False); axes[0][1].spines["right"].set_visible(False)
# 3. R vs G by city
for city,grp in df.sample(2000,random_state=42).groupby("city"):
    axes[1][0].scatter(grp["mean_r"],grp["mean_g"],alpha=0.35,s=10,color=CITY_CLR[city],label=city)
axes[1][0].set_xlabel("Mean R"); axes[1][0].set_ylabel("Mean G")
axes[1][0].set_title("Mean R vs G — City Clusters",fontweight="bold",loc="left")
axes[1][0].legend(fontsize=9); axes[1][0].spines["top"].set_visible(False); axes[1][0].spines["right"].set_visible(False)
# 4. R vs G by class
for lu,grp in df.sample(2000,random_state=42).groupby("land_use"):
    axes[1][1].scatter(grp["mean_r"],grp["mean_g"],alpha=0.35,s=10,color=CLASS_CLR[lu],label=lu)
axes[1][1].set_xlabel("Mean R"); axes[1][1].set_ylabel("Mean G")
axes[1][1].set_title("Mean R vs G — Class Clusters",fontweight="bold",loc="left")
axes[1][1].legend(fontsize=9); axes[1][1].spines["top"].set_visible(False); axes[1][1].spines["right"].set_visible(False)
plt.suptitle("Spectral EDA — VisionAI Lagos",fontweight="bold",y=1.01)
plt.tight_layout(); plt.savefig("outputs/spectral_eda.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

## Part 3 — PCA as a Feature Extractor

In [ ]:
# Build flat pixel feature matrix
print("Loading images as flat vectors (n=800)...")
sample=df.sample(800,random_state=42)
X_flat=[]; y_city=[]; y_class=[]
for _,row in tqdm(sample.iterrows(),total=len(sample)):
    arr=np.array(Image.open(IMG_DIR/row["filename"])).astype(float)/255.0
    X_flat.append(arr.flatten())
    y_city.append(row["city"]); y_class.append(row["land_use"])
X_flat=np.array(X_flat)
print(f"Feature matrix: {X_flat.shape}  (each image = {X_flat.shape[1]:,} values)")

scaler=StandardScaler(); X_s=scaler.fit_transform(X_flat)
pca=PCA(n_components=2,random_state=42)
X_pca=pca.fit_transform(X_s)
print(f"PCA explained variance: PC1={pca.explained_variance_ratio_[0]:.1%} PC2={pca.explained_variance_ratio_[1]:.1%}")

fig,axes=plt.subplots(1,2,figsize=(14,6))
for ax,(labels,clr_map,title) in zip(axes,[
    (y_city, CITY_CLR, "Cities — do city clusters separate cleanly?"),
    (y_class,CLASS_CLR,"Classes — do class clusters separate cleanly?")]):
    for label,colour in clr_map.items():
        mask=[i for i,l in enumerate(labels) if l==label]
        ax.scatter(X_pca[mask,0],X_pca[mask,1],alpha=0.35,s=10,color=colour,label=label)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
    ax.set_title(title,fontweight="bold",loc="left")
    ax.legend(fontsize=9,markerscale=2)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.suptitle("PCA Projection — City vs Class Separability",fontweight="bold")
plt.tight_layout(); plt.savefig("outputs/pca_spectral.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

In [ ]:
# PCA as classifier — evaluate at different k values
X_tr,X_te,y_tr,y_te=train_test_split(X_s,np.array(y_class),test_size=0.2,random_state=42,stratify=np.array(y_class))
k_values=[2,4,8,16,32,64,128]; f1s=[]
print(f"{'k':>5} {'Explained Var':>14} {'Weighted F1':>12}")
print("-"*35)
for k in k_values:
    pca_k=PCA(n_components=k,random_state=42)
    X_tr_k=pca_k.fit_transform(X_tr); X_te_k=pca_k.transform(X_te)
    lr=LogisticRegression(max_iter=1000,random_state=42); lr.fit(X_tr_k,y_tr)
    f1=f1_score(y_te,lr.predict(X_te_k),average="weighted"); f1s.append(f1)
    print(f"  {k:>3}  {pca_k.explained_variance_ratio_.sum():>13.1%}  {f1:>12.3f}")

fig,ax=plt.subplots(figsize=(8,5))
ax.plot(k_values,f1s,marker="o",color="#2E75B6",linewidth=2)
ax.set_title("PCA + LR: Weighted F1 vs k Components\n(diminishing returns beyond k≈32)",fontweight="bold",loc="left")
ax.set_xlabel("PCA components (k)"); ax.set_ylabel("Weighted F1"); ax.set_xscale("log"); ax.grid(axis="y",alpha=0.3)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.savefig("outputs/pca_elbow.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()
print(f"\nPCA baseline: spectral+GLCM ≈ 0.60-0.70, PCA k=32 ≈ {f1s[4]:.3f}")
print("CNN (Week 5) must beat these to justify its complexity.")

## Part 4 — Train/Val/Test Integrity Check

In [ ]:
print("=== SPLIT INTEGRITY CHECK ===\n")
for stat in ["brightness","ndvi_proxy","mean_r","mean_g"]:
    tr=df[df["split"]=="train"][stat].mean()
    va=df[df["split"]=="val"][stat].mean()
    te=df[df["split"]=="test"][stat].mean()
    flag="✅" if abs(tr-va)<5 and abs(tr-te)<5 else "⚠️"
    print(f"  {flag} {stat:<15}: train={tr:.1f}  val={va:.1f}  test={te:.1f}")
print("\nStratified by city × class — distributions should match across splits.")

## ⚠️ Common Mistakes — Week 2

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Fitting PCA on full dataset | Test leaks into components | Fit on train only; transform test |
| Raw pixel values for PCA without scaling | High-variance channels dominate | `StandardScaler()` before PCA |
| Choosing k by explained variance alone | May not maximise F1 | Plot F1 vs k — use the elbow |
| Treating BoVW clusters as CNN features | BoVW uses hand-crafted SIFT; CNN learns | Both cluster features, but differently |

## 🏆 Challenge Task

> Compute the **Bhattacharyya distance** between city brightness distributions. Which two cities are most spectrally similar? Formula: `B(p,q) = -ln(Σ√(p_i × q_i))`. Use `np.histogram` to compute discrete distributions.
>
> Hypothesis: the Bhattacharyya distance between Lagos and Kano predicts Week 9's cross-city F1 gap better than any single spectral statistic.

## ✅ What You Can Do After This Week

- Run a full EDA pipeline for image data
- Distinguish city-level from class-level spectral variation
- Use PCA as a feature extractor and evaluate vs k
- Verify train/val/test split integrity
- Explain BoVW as the image equivalent of TF-IDF

---
## ✅ Week 2 Complete — Next: `week3_preprocessing_STARTER.ipynb`

---
*The Computer Vision Internship · VisionAI Lagos · github.com/InternshipInABook*

## ✅ By completing Week 2, you can now:

- Run a complete EDA pipeline on a satellite image dataset
- Verify train/val/test split integrity — no leakage, balanced classes
- Use PCA to reduce image features and visualise class separability
- Write a one-paragraph EDA summary a product manager could act on

📤 **GitHub:** Push week2_eda_STARTER.ipynb. Commit: "Week 2: Full EDA complete, class separability measured"


---

## 📚 Get the Full Book

This notebook is part of **The Computer Vision Internship** — Book 3 of the InternshipInABook™ Series.

👉 **[Get the book on Selar → [SELAR_LINK_PLACEHOLDER]]**

---
*InternshipInABook™ · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook/cv-internship-in-a-book)*
